# Level 0: Understanding the Need for Asynchronous Programming

- below is a code going the the requests one by one(synchronously)

In [ ]:
import requests
import time

urls = ["http://example.com",
        "http://example.org",
        "https://www.google.com/search?q=valorant&oq=&gs_lcrp=EgZjaHJvbWUqCQgBEEUYOxjCAzIJCAAQRRg7GMIDMgkIARBFGDsYwgMyCQgCEEUYOxjCAzIJCAMQRRg7GMIDMgkIBBBFGDsYwgMyCQgFEEUYOxjCAzIJCAYQRRg7GMIDMgkIBxBFGDsYwgMyCQgIEEUYOxjCAzIJCAkQRRg7GMIDMgkIChBFGDsYwgMyCQgLEEUYOxjCAzIJCAwQRRg7GMIDMgkIDRBFGDsYwgMyCQgOEEUYOxjCA9IBBi0xajBqN6gCD7ACAQ&client=tablet-android-samsung-ss&sourceid=chrome-mobile&ie=UTF-8",]

start_time = time.time()

for url in urls:
    response = requests.get(url)
    print(response.status_code)

print(f"Sync code cost {time.time() - start_time:.2f} seconds")

200
200
200
Sync code cost 0.48 seconds


In [ ]:
import aiohttp
import asyncio # I/O bound
import time

async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            print(f"Status: {response.status}")

async def main():
    urls = ["http://example.com",
            "http://example.org",
            "https://www.google.com/search?q=valorant&oq=&gs_lcrp=EgZjaHJvbWUqCQgBEEUYOxjCAzIJCAAQRRg7GMIDMgkIARBFGDsYwgMyCQgCEEUYOxjCAzIJCAMQRRg7GMIDMgkIBBBFGDsYwgMyCQgFEEUYOxjCAzIJCAYQRRg7GMIDMgkIBxBFGDsYwgMyCQgIEEUYOxjCAzIJCAkQRRg7GMIDMgkIChBFGDsYwgMyCQgLEEUYOxjCAzIJCAwQRRg7GMIDMgkIDRBFGDsYwgMyCQgOEEUYOxjCA9IBBi0xajBqN6gCD7ACAQ&client=tablet-android-samsung-ss&sourceid=chrome-mobile&ie=UTF-8",]
    start_time = time.time()

    # 개체를 각각 생성해서 입력값으로 넣어준다.
    # 그럼 병렬적으로? 실행 해준다
    await asyncio.gather(*(fetch_url(url) for url in urls))
    print(f"Async code cost {time.time() - start_time:.2f} seconds")

#asyncio.run(main())

await main()

Status: 200
Status: 200
Status: 200
Async code cost 0.08 seconds


# Understanding the Event loop

- master scheduler
- enables non-blocking (not waiting for the process in front to end) executions possible

In [ ]:
import asyncio
import time

async def task_1():
    start_time = time.time()
    print("Starting task 1")
    # awaiting the sleep to end!
    await asyncio.sleep(2)
    end_time = time.time()
    print("Task 1 done")
    print(f"Task 1 duration: {end_time - start_time}")

async def task_2():
    start_time = time.time()
    print("Starting task 2")
    await asyncio.sleep(1)
    end_time = time.time()
    print("Task 2 done")
    print(f"Task 2 duration: {end_time - start_time}")

async def main():
    start_time = time.time()
    await asyncio.gather(task_1(), task_2())
    end_time = time.time()
    print(f"Total duration = {end_time - start_time}")

#asyncio.run(main())

await main()

Starting task 1
Starting task 2
Task 2 done
Task 2 duration: 1.0016958713531494
Task 1 done
Task 1 duration: 2.0039329528808594
Total duration = 2.0052778720855713


# Level 2: using async and await skillfully

`async`: defines a coroutine that is allowed to run with other operations, waits for the process defined behind `await`.  
 it will make a function `run` taking into account other coroutines!

`await`: while waiting, other coroutines will be running in the meantime. Use it for "another coroutine", which is awaitable.

- 그냥 프로세서를 넘겨줄 뿐, single thread인가?

# Level 3: Manage coroutines

`asyncio.gather()`: run many coroutines concurrently  
`asyncio.create_task()`: explicitly manage the tasks  

In [ ]:
import asyncio

async def task_1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 1 finished")
    return "Result 1"

async def task_2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 finished")
    return "Result 2"

# async 함수들은 그냥 syncrhonous 일반 함수들 내에서 아예 정의를 못하게 되어있다!
async def main(): #
    # 여기에 await를 붙이는 이유
    # gather 가 일단 두개를 concurrent하게 돌려버리고, 안 기다린다!!
    await asyncio.gather(task_1(), task_2())

# main() 안됨
# async 로 coroutine이 된 함수들은 await를 붙여야만 실행이된다!! 없으면 그냥 object
# 그리고 main()도 async로 정의했기 때문에 await 가 필요하다
await main()

Task 1 started
Task 2 started
Task 2 finished
Task 1 finished


In [ ]:
# code with create_task

import asyncio

async def task_1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 1 finished")
    return "Result 1"

async def task_2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 finished")
    return "Result 2"

async def main():
    t1 = asyncio.create_task(task_1()) # task를 스케쥴에 넣고 시작시킨다
    t2 = asyncio.create_task(task_2()) # task를 스케쥴에 넣고 시작시킨다

    # await를 사용하지 않으면 await asyncio.sleep 처럼 다음으로 오는 await를 실행하지 않고 끝낸다!
    # 한번 실행하면 기다려줄 필요가 없다는 뜻!
    await t1
    await t2

await main()

Task 1 started
Task 1 finished
Task 2 started
Task 2 finished


In [ ]:
# code with create_task

import asyncio

async def task_1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 1 finished")
    return "Result 1"

async def task_2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 finished")
    return "Result 2"

async def main():
    # 뭐가 문제인가? 직접 await를 한줄씩 작성하는 순간, 그냥 concurrent start가 안된다!!
    # create_task로 동시에 시작할 task들을 지정해준다고 생각할 수 있다!
    # "scheduling" 이 되어야 concurrent start 가 된다!
    await task_1()
    await task_2()

await main()

Task 1 started
Task 1 finished
Task 2 started
Task 2 finished


# Level 4: Cancel Async Tasks that are too-long

`cancel()`: direct cancellation of task

In [ ]:
import asyncio

async def task_1():
    print("Task 1 started")
    await asyncio.sleep(0.5)
    print("t1 slept for 0.5")
    await asyncio.sleep(0.45) # the order with task_1 is quite random
    print("t1 slept for another 0.45")

    # 이쯤에서 cancel이 request를 날렸다
    # task_1()이 뭔가를 실행하는 순간, 그때서야 asyncio.CancelledError 를 생성하게 된다.

    try:
        await asyncio.sleep(1) # 이걸 시도하는 순간 asyncio.CancelledError 를 생성
    except asyncio.CancelledError: # 그래서 이 코드로 넘어온다.
        print("Task 1 was cancelled")
        raise
    print("Task 1 finished")
    return "Result 1"

async def task_2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 finished")
    return "Result 2"

async def main():
    t1 = asyncio.create_task(task_1())
    t2 = asyncio.create_task(task_2())

    # sleeps for 1 second
    await t2
    # t1 will also be sleeping for 1 second

    # after t2 has ended, t1 reaches a cancel()
    t1.cancel() # .cancel in fact "sets a flag" on t1
    # the t1 task will then raise asyncio.CancelledError!

    try:
        await t1 # 이미 task_1에서 asyncio.CancelledError를 잘 처리했지만, await로 또 불러오는 순간 다시 해당 Error가 발생!!
    except asyncio.CancelledError: # re-raise 된 error를 이제 또 처리하는 것이다!
        print("Handled cancellation of Task 1")

await main()

Task 1 started
Task 2 started
t1 slept for 0.5
t1 slept for another 0.45
Task 2 finished
Task 1 was cancelled
Handled cancellation of Task 1


# Level 5: Timeout Tasks Handling with `asyncio.wait_for()`

얼만큼 기다릴 것인지 시간을 지정할 수 있다.

In [ ]:
import asyncio

async def slow_task():
    await asyncio.sleep(5)
    return "Task finished"

async def main():
    try:
        result = await asyncio.wait_for(slow_task(), timeout=2) # 2초간 기다리다가 넘어가면 TimeoutError 에 대한 request를 날려준다!
        print(result)
    except asyncio.TimeoutError:
        print("Task timed out!")

await main()

Task timed out!


# Level 6: Limiting Concurrency with asyncio.Semaphore to prevent resources overload

- 한꺼번에 너무 많이 돌리면 리소스를 다 잡아먹을 수 있다.
- 동시에 돌릴 프로세스의 개수를 제한할 수 있다.

- thread는 어차피 한개이지만, 서버 연결 이나 request개수가 아주 많아지면 관리 해야되는 프로세스가 많아지면서 RAM에 올라가야 프로세스가 많아질 수 있다.

In [ ]:
import asyncio

semaphore = asyncio.Semaphore(10) # 5개로 프로세스 개수 제한

async def limited_task(n):
    async with semaphore: # 제한 5개를 가지고 한다는 뜻
        print(f'Task {n} started')
        await asyncio.sleep(1)
        print(f'Task {n} finished')

async def main():
    tasks = (limited_task(i) for i in range(500)) # 총 task의 개수는 10개,
    await asyncio.gather(*tasks) # 한번에 5개의 프로세스만 schedule에 올라가게 되었다!!!

await main()

# 숫자를 크게 바꾸니까 실제로 RAM 사용량이 치솟는다
# tasks 를 한번에 모두 생성하는 것이 아닌 generator로 해보자

Task 0 started
Task 1 started
Task 2 started
Task 3 started
Task 4 started
Task 5 started
Task 6 started
Task 7 started
Task 8 started
Task 9 started
Task 10 started
Task 11 started
Task 12 started
Task 13 started
Task 14 started
Task 15 started
Task 16 started
Task 17 started
Task 18 started
Task 19 started
Task 0 finished
Task 1 finished
Task 2 finished
Task 3 finished
Task 4 finished
Task 5 finished
Task 6 finished
Task 7 finished
Task 8 finished
Task 9 finished
Task 10 finished
Task 11 finished
Task 20 started
Task 21 started
Task 22 started
Task 23 started
Task 24 started
Task 25 started
Task 26 started
Task 27 started
Task 28 started
Task 29 started
Task 30 started
Task 31 started
Task 12 finished
Task 13 finished
Task 14 finished
Task 15 finished
Task 16 finished
Task 17 finished
Task 18 finished
Task 19 finished
Task 32 started
Task 33 started
Task 34 started
Task 35 started
Task 36 started
Task 37 started
Task 38 started
Task 39 started
Task 20 finished
Task 21 finished
Task 2

# Level 7: Error Handling for Asynchronous

- 주의점: async를 사용할 때는 반드시 해당 함수들 내에서 에러를 처리하게끔 만들어야 한다!! concurrent 를 잘 관리해야 한다.

In [ ]:
import asyncio

async def task_1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 2 finished")
    return "Result 1"

async def main():
    t1 = asyncio.create_task(task_1())
    t1.cancel()
    try:
        await t1
    except asyncio.CancelledError:
        print("Handled cancellation of task 1")

await main()
# 그냥 끝

Handled cancellation of task 1


# Level 8: Async queue

queue: producer-consumer pattern, acts as a buffer between them

python has `queue.Queue` and async has `asyncio.Queue`

In [ ]:
import asyncio

async def producer(queue):
    for i in range(3): # 3개의 아이템을 put으로 입력했다
        print(f"Producing {i}")
        await queue.put(i) # 웹의 http put 요청 맞음, json 만들고 업데이트 하는등
        # 아, 이렇게 기다리는 상태가 아무리 작은 숫자여도 존재하는 순간 이 순간에 queue.get()이 실행되기 시작된다!!!
        await asyncio.sleep(0.00000001)

async def consumer(queue):
    while True: # 얘가 첫번째로 실행이 되는 코드 블록인데, 멈추지 않으므러 이 안에 있는 await가 다 실행이 된다...
        item = await queue.get() # get 요청 맞음, 조회하는 등
        print(f"Consuming {item}") # await가 없는 코드들은 즉시 실행이다!
        await asyncio.sleep(1)
        queue.task_done() # 여기까지 다 실행하고, while문의 await queue.get() 를 다시 맞닥뜨리면 다시 prod로 간다!

async def main():
    queue = asyncio.Queue() # queue creation
    prod = asyncio.create_task(producer(queue)) # prod에 넣어주고, # create_task 만으로도 initialize는 된다!! concurrent
    cons = asyncio.create_task(consumer(queue)) # cons에도 넣어주었다. # 얘도 이제 동시에 돌아간다.

    await asyncio.gather(prod) #await를 통해서 main()이 prod가 끝나기까지 기다리게 만들었다!!!
    await queue.join() # put()이 무조건 task_done()으로 끝날때까지 기다린다. task_done()이 cons에 있으므로, cons를 기다린다!
    # 시간적으로 자연스럽게 없어도 되지만, 확실하게 만들어준다. prevents premature termination
    # cons으 sleep이 너무 길어지니까 producing이 먼저 되었지만, 그래도 get()이 다 오도록 기다리게 만들어주었다!!!
    cons.cancel() #이 cancel에 전부 실행되기도 전에 맞닥뜨릴수 있기 때문에 queue.join() 이 필용하다

await main()

Producing 0
Consuming 0
Producing 1
Producing 2
Consuming 1
Consuming 2
